# 01 · Building the country-level dataset

**Pipeline position:** `Data cleaning` → Standardization → Correlation → PCA → Clustering → Interpretation

This notebook produces one clean table: **one row per country, 11 socioeconomic indicators for one year**, plus
the World Bank labels (`region`, `income_group`, `lending_type`) kept *only* for interpretation.

| Step | What happens |
|---|---|
| A | Download country metadata from the World Bank API and **remove aggregates** (World, "High income", regions…) |
| B | Load the WDI bulk CSV and keep only the 11 indicators |
| C | Compare data coverage across years and **choose the year *t*** |
| D | Pivot to a country × indicator table and merge the metadata |
| E | **Report and analyse missingness** (by variable, by country, by income group / region) |
| F | Decide which countries to drop (rule + justification) |
| G | Impute the remaining gaps (method validated, no labels used) |
| H | Save the processed tables and a `dataset_info.json` for reproducibility |

> **Before you run this:** download the WDI bulk CSV zip from <https://datatopics.worldbank.org/world-development-indicators/>
> (section *Data → Bulk download → CSV*), unzip it, and put `WDICSV.csv` (or the whole unzipped folder) inside `data/raw/`.

In [ ]:
import sys, json, datetime
from pathlib import Path

# Find the project root (the folder that contains src/common.py), wherever Jupyter starts.
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "common.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Project root not found. In VS Code use File > Open Folder... and open the Project folder.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from matplotlib.colors import ListedColormap
from scipy import stats
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer

from src.common import INDICATORS, FEATURES, INCOME_ORDER

RAW, PROC, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports" / "figures"
for p in (RAW, PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

def savefig(name, **kw):
    plt.tight_layout(**kw)
    plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

print("Project root:", ROOT)

## Step A · Country metadata (region, income group) and removal of aggregates

The World Bank API returns ~296 "entities". About 80 of them are **aggregates** (World, regional totals, income-group
totals…). The API marks them with `region = "Aggregates"`, so they are easy to filter out.
The response is cached in `data/raw/wb_country_api.json` so the notebook also works offline later
(delete that file if you want to refresh it).

In [ ]:
META_URL = "https://api.worldbank.org/v2/country?format=json&per_page=400"
META_CACHE = RAW / "wb_country_api.json"

if META_CACHE.exists():
    payload = json.loads(META_CACHE.read_text(encoding="utf-8"))
    print("Loaded cached API response:", META_CACHE.name)
else:
    resp = requests.get(META_URL, timeout=60)
    resp.raise_for_status()
    payload = resp.json()
    META_CACHE.write_text(json.dumps(payload), encoding="utf-8")
    print("Downloaded from the World Bank API and cached to", META_CACHE.name)

page_info, records = payload
assert page_info["pages"] == 1, f"API returned {page_info['pages']} pages - increase per_page in META_URL"

meta_all = pd.DataFrame([{
    "code": c["id"],
    "name": c["name"],
    "region": c["region"]["value"].strip(),
    "income_group": c["incomeLevel"]["value"].strip(),
    "lending_type": c["lendingType"]["value"].strip(),
} for c in records])

is_agg = meta_all["region"].eq("Aggregates")
print(f"API entities: {len(meta_all)} | aggregates removed: {is_agg.sum()} | countries/economies kept: {(~is_agg).sum()}")

meta = meta_all.loc[~is_agg].set_index("code").sort_index()
display(meta["income_group"].value_counts().rename("n_economies").to_frame())
display(meta["region"].value_counts().rename("n_economies").to_frame())

**Check:** the income-group table should contain Low / Lower middle / Upper middle / High income (and possibly a few
"Not classified" economies). Those unclassified economies stay in the clustering data set but are left out of the
income-group *comparisons* later.

## Step B · Load the WDI bulk CSV and keep only the 11 indicators

The full file is large (hundreds of MB), so it is read in chunks and filtered on the fly.

In [ ]:
csv_path = next(RAW.rglob("WDICSV.csv"), None)
if csv_path is None:
    raise FileNotFoundError(
        "WDICSV.csv not found under data/raw/. Download the WDI CSV zip from "
        "https://datatopics.worldbank.org/world-development-indicators/ (Bulk download -> CSV), "
        "unzip it, and place WDICSV.csv (or the unzipped folder) in data/raw/."
    )
print("Using:", csv_path.relative_to(ROOT), f"({csv_path.stat().st_size / 1e6:.0f} MB)")

ID_COLS = {"Country Name", "Country Code", "Indicator Name", "Indicator Code"}
parts, wdi_codes = [], set()
for chunk in pd.read_csv(csv_path, usecols=lambda c: c in ID_COLS or c.strip().isdigit(), chunksize=250_000):
    wdi_codes |= set(chunk["Country Code"].unique())
    parts.append(chunk[chunk["Indicator Code"].isin(INDICATORS)])
sub_all = pd.concat(parts, ignore_index=True)

absent = set(INDICATORS) - set(sub_all["Indicator Code"])
assert not absent, f"These indicator codes were not found in the CSV: {absent}"

# Reconcile the two sources (WDI file vs. API metadata)
print("WDI entities not known to the API  :", sorted(wdi_codes - set(meta_all["code"])) or "none")
print("API countries absent from WDI file :", sorted(set(meta.index) - wdi_codes) or "none")

# keep real countries only; one row per (country, indicator)
sub = sub_all[sub_all["Country Code"].isin(meta.index)]
assert not sub.duplicated(["Country Code", "Indicator Code"]).any(), "duplicate (country, indicator) rows"
year_cols = [c for c in sub.columns if c.strip().isdigit()]
wide = sub.set_index(["Country Code", "Indicator Code"])[year_cols]
print(f"Rows kept: {len(sub):,} | years in file: {year_cols[0]}-{year_cols[-1]}")

## Step C · Choosing the year *t*

Recent years often have gaps because many indicators are published with a lag (electricity access, school enrolment and
unemployment in particular). For each candidate year we count how many countries have a (nearly) complete profile.
Choose a year with **good coverage but still recent** — and write the justification in your report.

In [ ]:
CAND_YEARS = [y for y in year_cols if int(y) >= 2015]
all_codes = sorted(meta.index)
rows, ind_cov = [], {}
for y in CAND_YEARS:
    piv = wide[y].unstack("Indicator Code").reindex(index=all_codes, columns=list(INDICATORS))
    n_miss = piv.isna().sum(axis=1)
    rows.append({"year": y, "complete_all_11": int((n_miss == 0).sum()),
                 "<=3_missing": int((n_miss <= 3).sum()),
                 "avg_%_missing": round(piv.isna().mean().mean() * 100, 1)})
    ind_cov[y] = piv.notna().mean() * 100

cov_table = pd.DataFrame(rows).set_index("year")
display(cov_table)

ind_cov = pd.DataFrame(ind_cov).T.rename(columns=INDICATORS)     # rows = years, cols = indicators
fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(ind_cov, annot=True, fmt=".0f", cmap="YlGn", vmin=0, vmax=100,
            cbar_kws={"label": "% of countries with a value"}, ax=ax)
ax.set_title("Data coverage by year and indicator"); ax.set_ylabel("")
savefig("01_coverage_by_year")

## Step D · Build the country × indicator table for year *t* and merge metadata

In [ ]:
YEAR = "2022"     # <-- decision: edit if the coverage table above suggests a better year, then re-run

df_year = (wide[YEAR].unstack("Indicator Code")
           .reindex(index=all_codes, columns=list(INDICATORS))
           .rename(columns=INDICATORS))

data = meta[["name", "region", "income_group", "lending_type"]].join(df_year)   # labels + 11 features
data.index.name = "code"

print(f"Year {YEAR}: {data.shape[0]} countries x {len(FEATURES)} indicators (+ 3 label columns)")
display(data.head())

## Step E · Missingness analysis (report first, decide later)

We look at missingness three ways: per **variable**, per **country**, and per **group** (income group / region).
The last one matters: if missingness is concentrated in, say, low-income countries, then simply deleting rows would
silently remove exactly the countries we want to cluster.

In [ ]:
miss = data[FEATURES].isna()
n_miss = miss.sum(axis=1)

var_tbl = pd.DataFrame({"n_missing": miss.sum(), "pct_missing": (miss.mean() * 100).round(1)}) \
            .sort_values("pct_missing", ascending=False)
display(var_tbl)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(x=var_tbl["pct_missing"], y=var_tbl.index, color="#4575b4", ax=axes[0])
axes[0].set_xlabel("% of countries with a missing value"); axes[0].set_ylabel(""); axes[0].set_title("Missingness per variable")

cnt = n_miss.value_counts().sort_index()
sns.barplot(x=cnt.index, y=cnt.values, color="#fc8d59", ax=axes[1])
axes[1].set_xlabel("number of missing indicators (out of 11)"); axes[1].set_ylabel("number of countries")
axes[1].set_title("Missingness per country")
savefig("01_missingness_overview")

# Which countries are worst? (most missing first)
worst = data.assign(n_missing=n_miss, missing_vars=[", ".join(c for c in FEATURES if miss.loc[i, c]) for i in data.index])
display(worst.sort_values("n_missing", ascending=False)[["name", "income_group", "n_missing", "missing_vars"]].head(25))

In [ ]:
# Heatmap of every cell: rows sorted by income group, then by number of missing values
rank = data["income_group"].map({g: i for i, g in enumerate(INCOME_ORDER)}).fillna(len(INCOME_ORDER))
order = pd.DataFrame({"rank": rank, "n": n_miss}).sort_values(["rank", "n"]).index

fig, ax = plt.subplots(figsize=(9, 14))
sns.heatmap(miss.loc[order].astype(int), cmap=ListedColormap(["#eeeeee", "#b2182b"]), cbar=False,
            yticklabels=False, ax=ax)
for edge in rank.loc[order].value_counts(sort=False).sort_index().cumsum().values[:-1]:
    ax.axhline(edge, color="black", lw=1)
ax.set_title("Missing cells (red) - rows grouped Low -> High income (black lines)\n")
ax.set_ylabel("countries"); ax.tick_params(axis="x", rotation=45)
savefig("01_missingness_heatmap")

In [ ]:
# Is missingness related to income group / region?  (relevant for choosing a fair strategy)
grp_order = [g for g in INCOME_ORDER if g in set(data["income_group"])] + \
            sorted(set(data["income_group"]) - set(INCOME_ORDER))
by_income = (miss.groupby(data["income_group"]).mean() * 100).reindex(grp_order)
by_region = (miss.groupby(data["region"]).mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sns.heatmap(by_income, annot=True, fmt=".0f", cmap="Reds", vmin=0, vmax=60, ax=axes[0],
            cbar_kws={"label": "% missing"})
axes[0].set_title("% missing by income group"); axes[0].set_ylabel("")
sns.heatmap(by_region, annot=True, fmt=".0f", cmap="Reds", vmin=0, vmax=60, ax=axes[1],
            cbar_kws={"label": "% missing"})
axes[1].set_title("% missing by region"); axes[1].set_ylabel("")
savefig("01_missingness_by_group")

groups = [n_miss[data["income_group"] == g].values for g in INCOME_ORDER]
H, p = stats.kruskal(*[g for g in groups if len(g) > 0])
print(f"Kruskal-Wallis test, #missing indicators across income groups: H = {H:.2f}, p = {p:.4f}")
display(n_miss.groupby(data["income_group"]).agg(["count", "mean", "max"]).reindex(grp_order).round(2))

**Your observations (write in the report):** Which variables are the most incomplete? Which kinds of countries
(income group / region / small states) have the most gaps? Is the missingness plausibly *random*, or does it depend on
the country's development level? What does that imply for deleting versus imputing?

## Step F · Which countries to drop?

**Rule used here:** drop a country if more than `MAX_MISSING` of its 11 indicators are missing.

*Justification:* a country with 4 or more missing values has over a third of its profile unobserved, so imputing it
would mean most of its cluster-defining information is invented from other countries. Below that threshold the
imputation is anchored on a mostly observed profile. The sensitivity table shows how many countries each possible
threshold would keep, so you can defend (or change) the choice.

In [ ]:
MAX_MISSING = 3     # <-- decision: drop countries with MORE than this many missing indicators

sens = pd.DataFrame({"max missing allowed": range(0, 9)})
sens["countries kept"] = [(n_miss <= t).sum() for t in sens["max missing allowed"]]
sens["countries dropped"] = len(data) - sens["countries kept"]
display(sens.set_index("max missing allowed").T)

drop_mask = n_miss > MAX_MISSING
kept = data.loc[~drop_mask].copy()
dropped = data.loc[drop_mask, ["name", "region", "income_group"]].copy()
dropped["n_missing"] = n_miss[drop_mask]
dropped["missing_vars"] = [", ".join(c for c in FEATURES if miss.loc[i, c]) for i in dropped.index]
print(f"Kept {len(kept)} countries, dropped {len(dropped)} (more than {MAX_MISSING} missing indicators).")
display(dropped.sort_values("n_missing", ascending=False))

# Does dropping change the composition of the sample? (fairness check)
comp = pd.DataFrame({"all": data["income_group"].value_counts(),
                     "kept": kept["income_group"].value_counts(),
                     "dropped": dropped["income_group"].value_counts()}).fillna(0).astype(int)
comp["% dropped"] = (comp["dropped"] / comp["all"] * 100).round(0)
display(comp.reindex(grp_order))

worst_var = (kept[FEATURES].isna().mean() * 100).sort_values(ascending=False)
print("\nRemaining missingness among kept countries (% per variable):")
display(worst_var.round(1).to_frame("pct_missing").T)
if worst_var.iloc[0] > 40:
    print("WARNING: a variable is still >40% missing - consider another year or dropping that variable.")

## Step G · Imputing the remaining gaps

**Choice: KNN imputation (k = 5, distance-weighted)** on robustly-scaled data, with GDP per capita handled on the log10
scale. *Reasons:* (i) the indicators are strongly inter-correlated (income, health, infrastructure), so a country's
nearest neighbours in indicator space predict its missing values far better than a global median; (ii) it preserves the
joint structure that clustering and PCA depend on; (iii) it uses **only the 11 indicators** — never `income_group` or
`region`, which would leak the labels we want to evaluate against.

Rather than assert that KNN is better, we **test it**: take the complete-case countries, hide 15 % of the cells at random,
impute them, and compare the error with simple median imputation.

In [ ]:
def impute_knn(frame, k=5):
    sc = RobustScaler()                                   # median/IQR: not dominated by extreme inflation values
    Z = sc.fit_transform(frame)                           # NaNs are ignored when fitting and preserved
    Zi = KNNImputer(n_neighbors=k, weights="distance").fit_transform(Z)
    return pd.DataFrame(sc.inverse_transform(Zi), index=frame.index, columns=frame.columns)

def impute_median(frame):
    return frame.fillna(frame.median())

work = kept[FEATURES].copy()
work["GDP_PC"] = np.log10(work["GDP_PC"])                 # impute GDP per capita on the log scale

complete = work.dropna()
print(f"Complete-case countries available for the hold-out test: {len(complete)}")
if len(complete) < 30:
    print("WARNING: very few complete cases - the validation below will be noisy.")

methods = {"median": impute_median,
           "KNN k=3": lambda f: impute_knn(f, 3),
           "KNN k=5": lambda f: impute_knn(f, 5),
           "KNN k=10": lambda f: impute_knn(f, 10)}

rng = np.random.default_rng(SEED)
sd = complete.std().values
scores = {m: [] for m in methods}
for rep in range(30):
    mask = rng.random(complete.shape) < 0.15
    test = complete.mask(mask)
    for m, fn in methods.items():
        err = ((fn(test).values - complete.values) / sd) ** 2
        scores[m].append(np.sqrt(err[mask].mean()))

res = pd.DataFrame(scores)
display(pd.DataFrame({"mean RMSE (in std. units)": res.mean(), "std over 30 repeats": res.std()}).round(3))

**Reading the table:** lower RMSE = the imputed values are closer to the true (hidden) ones. If KNN clearly beats the
median, KNN is justified; quote these numbers in your report. Pick `IMPUTE_K` below accordingly.

In [ ]:
IMPUTE_K = 5      # <-- decision (see the validation table above)

imp = impute_knn(work, IMPUTE_K)
imp["GDP_PC"] = 10 ** imp["GDP_PC"]                       # back to US$

mask_final = kept[FEATURES].isna()                        # which cells were missing
filled = kept[FEATURES].mask(mask_final, imp)             # observed values stay EXACTLY as they were
assert filled.notna().all().all(), "NaNs remain after imputation"

print(f"Imputed {int(mask_final.values.sum())} cells out of {mask_final.size} "
      f"({mask_final.values.mean() * 100:.1f} %) across {int(mask_final.any(axis=1).sum())} countries.")
display(pd.DataFrame({"cells imputed": mask_final.sum(),
                      "mean before (observed only)": kept[FEATURES].mean(),
                      "mean after imputation": filled.mean()}).round(2))

## Step H · Save the processed tables

Two versions are saved on purpose:

* `countries_filtered.csv` – countries kept, **missing values left as NaN** (use for EDA, so distributions are not distorted by imputation);
* `countries_imputed.csv` – complete table (use for standardisation, PCA and clustering);
* `imputed_mask.csv` – 1 where a cell was imputed (useful later when interpreting "unusual" countries);
* `dropped_countries.csv`, `dataset_info.json` – documentation of what was done.

In [ ]:
labels = kept[["name", "region", "income_group", "lending_type"]]

pd.concat([labels, kept[FEATURES]], axis=1).to_csv(PROC / "countries_filtered.csv")
pd.concat([labels, filled], axis=1).to_csv(PROC / "countries_imputed.csv")
mask_final.astype(int).to_csv(PROC / "imputed_mask.csv")
dropped.to_csv(PROC / "dropped_countries.csv")

info = {
    "year": YEAR,
    "wdi_source_file": f"{csv_path.parent.name}/{csv_path.name}",
    "run_date": datetime.date.today().isoformat(),
    "n_economies_after_removing_aggregates": int(len(data)),
    "max_missing_allowed": MAX_MISSING,
    "n_countries_kept": int(len(kept)),
    "n_countries_dropped": int(len(dropped)),
    "imputation": f"KNN (k={IMPUTE_K}, distance-weighted) on RobustScaler-scaled indicators; GDP_PC imputed on log10 scale; labels not used",
    "n_cells_imputed": int(mask_final.values.sum()),
    "seed": SEED,
    "features": FEATURES,
}
(PROC / "dataset_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")

print(json.dumps(info, indent=2))
print("\nSaved to", PROC.relative_to(ROOT))
for f in sorted(PROC.glob("*")):
    print(" -", f.name)

## Done - what to commit and what comes next

```bash
git add notebooks/01_data_building.ipynb data/processed reports/figures
git commit -m "Build country dataset: missingness analysis, filtering, KNN imputation"
git push
```

**Next:** `02_eda.ipynb` (summary statistics, distributions, income-group comparison, correlations, scatter plots).